# Python Code Error Debugger with QLoRA Fine-tuning

Fine-tune Meta's **Llama 3.2 3B Instruct** model to debug Python code errors using real Stack Overflow Q&A data.

## Project Overview
---

This notebook fine-tunes an LLM to act as a Python code debugger. Given a broken piece of code and an optional error message, the model explains what went wrong and provides corrected code.

The fine-tuning uses **QLoRA (Quantized Low-Rank Adaptation)**, which makes it possible to train a 3B parameter model on a single GPU by quantizing the base model to 4-bit and only training a small set of adapter weights (~0.51% of total parameters).

## Technical Details
---

- **Base Model:** `meta-llama/Llama-3.2-3B-Instruct`  
- **Dataset:** `koutch/stackoverflow_python` (~987k Stack Overflow Python Q&A pairs)  
- **Training and eval samples:** 15,000 (quality-filtered from 431k error-related posts)  
- **Quantization:** 4-bit NF4 with double quantization via bitsandbytes  
- **LoRA rank:** r=16, targeting attention projection layers  
- **Effective batch size:** 32 (batch_size=8 × gradient_accumulation=4)  
- **Optimizer:** paged_adamw_8bit  
- **LR scheduler:** Cosine with warmup  

## Project Features
---

- Quality-filtered Stack Overflow dataset (error/exception keywords + code presence filter)
- QLoRA fine-tuning with 4-bit quantization, it trains on a single A100 in ~1.5 hrs
- Early stopping based on validation loss to prevent overfitting
- Best checkpoint selection via `load_best_model_at_end`


**Model:** `sarthakmasta/code-debugger-llama`

## Part 1. Setup and Installation

In [1]:
!pip install transformers accelerate peft bitsandbytes datasets trl gradio torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 44.2 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
import gradio as gr
from google.colab import userdata
from huggingface_hub import login
import re
from peft import PeftModel
from transformers import EarlyStoppingCallback

In [3]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

## Part 2. Loading and Preprocessing Dataset

In [4]:
data = load_dataset("koutch/stackoverflow_python", split="train")
print(len(data))

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00005-3d48b7eced91a0(…):   0%|          | 0.00/121M [00:00<?, ?B/s]

data/train-00001-of-00005-56eee9940d3dd0(…):   0%|          | 0.00/154M [00:00<?, ?B/s]

data/train-00002-of-00005-2683a3aec26c96(…):   0%|          | 0.00/175M [00:00<?, ?B/s]

data/train-00003-of-00005-dfcfc12a49338d(…):   0%|          | 0.00/186M [00:00<?, ?B/s]

data/train-00004-of-00005-69e77510bada75(…):   0%|          | 0.00/193M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/987122 [00:00<?, ? examples/s]

987122


In [5]:
def error_filter(sample):
  contents = sample['title'].lower() + " " + sample['question_body'].lower()
  keywords = ['error', 'exception', 'bug', 'traceback', 'fails', 'broken', 'doesnt work', "doesn't work", 'issue', 'problem']
  return any(keyword in contents for keyword in keywords)

filtered_data = data.filter(error_filter)
print(len(filtered_data))

def quality_filter(sample):
    answer = sample['answer_body']
    question = sample['question_body']

    # Must have a meaningful answer (not just a comment)
    if len(answer) < 150:
        return False

    # Answer should contain code (good debugging answers show corrected code)
    has_code = '<code>' in answer or '```' in answer or 'def ' in answer or 'print(' in answer
    if not has_code:
        return False

    # Question should contain code too (we want code+error pairs, not theory questions)
    question_has_code = '<code>' in question or '```' in question or 'def ' in question
    if not question_has_code:
        return False

    return True

filtered_data = filtered_data.filter(quality_filter)
print(f"After quality filter: {len(filtered_data)}")

Filter:   0%|          | 0/987122 [00:00<?, ? examples/s]

431829


Filter:   0%|          | 0/431829 [00:00<?, ? examples/s]

After quality filter: 326164


In [6]:
def format_instruction(sample):
    problem = sample['title']
    code_body = sample['question_body']
    solution = sample['answer_body']

    code_block = re.search(r'```python\n(.*?)\n```', code_body, re.DOTALL)
    code = code_block.group(1) if code_block else code_body[:500]

    instruction = f"""Debug this piece of Python code and explain the error.

Question: {problem}

Code:
```python
{code[:400]}
```

Provide:
1. Explanation of the error
2. Corrected code"""

    solution_clean = re.sub(r'<[^>]+>', '', solution)
    solution_clean = solution_clean.strip()
    solution_clean = re.sub(r'\n{3,}', '\n\n', solution_clean)
    response = solution_clean[:1200]

    prompt = f"""<|begin_of_text|><|start_header_id|>user<|end_header_id|>

{instruction}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{response}<|eot_id|>"""

    return {"text": prompt}

formatted_data = filtered_data.map(format_instruction)
formatted_data = formatted_data.remove_columns([col for col in formatted_data.column_names if col != 'text'])

Map:   0%|          | 0/326164 [00:00<?, ? examples/s]

In [7]:
target_size = 15000
formatted_data = formatted_data.shuffle(seed=42).select(range(target_size))
train_dataset = formatted_data.select(range(int(0.9 * len(formatted_data))))
eval_dataset = formatted_data.select(range(int(0.9 * len(formatted_data)), len(formatted_data)))

print(len(train_dataset))
print(len(eval_dataset))

13500
1500


## Part 3. Loading Base Model with QLoRA Configuration

In [8]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=quant_config, device_map="auto", trust_remote_code=True)

model = prepare_model_for_kbit_training(model)

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [9]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_parameters = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters = ({trainable_parameters / all_parameters * 100:.2f}%)")

Trainable parameters = (0.51%)


## Part 4. Training Configuration

In [11]:
training_args = SFTConfig(
    output_dir="./code-debugger-llama",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=5e-5,
    bf16=True,
    tf32=True,
    logging_steps=100,
    eval_steps=100,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    save_strategy="steps",
    save_steps=100,
    eval_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    max_length=768,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/13500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/13500 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

## Part 5. Training the Model

In [12]:
trainer.train()
print("Complete")
trainer.save_model("./code-debugger-final")
tokenizer.save_pretrained("./code-debugger-final")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss,Validation Loss
100,2.191642,1.842534
200,1.838405,1.811637
300,1.822361,1.797949
400,1.810462,1.790418
500,1.801955,1.785024
600,1.795798,1.781013
700,1.776003,1.777590
800,1.779930,1.775655
900,1.769846,1.774702
1000,1.773015,1.774324


Complete


('./code-debugger-final/tokenizer_config.json',
 './code-debugger-final/chat_template.jinja',
 './code-debugger-final/tokenizer.json')

In [15]:
del model
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto"
)

fine_tuned_model = PeftModel.from_pretrained(base_model, "./code-debugger-final")
fine_tuned_model = fine_tuned_model.merge_and_unload()

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

## Part 6. Pushing to HuggingFace Hub

In [16]:
fine_tuned_model.push_to_hub("sarthakmasta/code-debugger-llama")
tokenizer.push_to_hub("sarthakmasta/code-debugger-llama")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...3zl0l89/model.safetensors:   0%|          | 29.9MB / 6.43GB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp8lsc_v23/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

CommitInfo(commit_url='https://huggingface.co/sarthakmasta/code-debugger-llama/commit/c5341491f37b458475a63ee70e874af08d7f2c6c', commit_message='Upload tokenizer', commit_description='', oid='c5341491f37b458475a63ee70e874af08d7f2c6c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/sarthakmasta/code-debugger-llama', endpoint='https://huggingface.co', repo_type='model', repo_id='sarthakmasta/code-debugger-llama'), pr_revision=None, pr_num=None)

## Part 7. Testing and Gradio Interface

In [53]:
def debug_code(code, error_message=""):
    prompt = f"""<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Debug this piece of Python code and explain the error.

Question: {error_message if error_message.strip() else "What is wrong with this code?"}

Code:
```python
{code}
```

Provide:
1. Explanation:
Explanation of the error
2. Corrected code:
correct the whole entire input code block, not just one line<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = fine_tuned_model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.3,
        do_sample=True,
        top_p=0.9,
        repetition_penalty=1.1
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.split("assistant")[-1].strip()

    return response

In [54]:
test_code = """
def calculate_average(numbers):
    total = 0
    for num in numbers:
        total += num
    return total / len(numbers)

result = calculate_average([])
print(result)
"""

print(debug_code(test_code, "ZeroDivisionError: division by zero"))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


You're trying to divide by a list (which is an empty list), which will raise a ZeroDivisionError. You should check if the list is empty before you try to divide it:

def calculate_average(numbers):
    if len(numbers) == 0:
        return None
    else:
        total = 0
        for num in numbers:
            total += num
        return total / len(numbers)

result = calculate_average([])
print(result)

This way, when you pass an empty list to your function, it will simply return None instead of raising an exception.


In [55]:
def gradio_debug(code_input, error_input):
    if not code_input.strip():
        return "Please provide code to debug."
    return debug_code(code_input, error_input)

with gr.Blocks() as ui:
    gr.Markdown("# Python Code Debugger")

    code_input = gr.Code(label="Your Python Code", language="python", lines=12)
    error_input = gr.Textbox(label="Error Message (Optional)", placeholder="Paste error traceback here...", lines=3)
    output = gr.Markdown(label="Debug Output")

    debug_button = gr.Button("Debug Code")
    debug_button.click(fn=gradio_debug, inputs=[code_input, error_input], outputs=output)

ui.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://983f7d4ebba80966c0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
